# 8-bob. 8-BOB. ZARARLI DASTURLARNI ANIQLASH II: DINAMIK TAHLIL VA CHUQUR O'QITISH
## Amaliy misol: API chaqiruvlari bo'yicha tasniflash

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/08-bob/8-1_api-chaqiruvlari-boyicha-tasniflash.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 8-bob.*

Quyidagi dasturda 2000 ta jarayonning API chaqiruvlari izi sun'iy ravishda yaratiladi. Zararsiz va zararli jarayonlar bir xil umumiy funksiyalardan foydalanadi. Zararli jarayonlarda yuqorida tasvirlangan kod kiritish zanjiri aniq tartibda uchraydi. Zararsiz jarayonlarning 30 foizi (tuzatuvchi dasturlarni modellashtiradi) ham xuddi shu to'rtta funksiyani chaqiradi, lekin boshqa tartibda. Bundan tashqari, avtoyuklash bo'limiga yozish ham ikkala guruhda uchraydi. Ikki xil model taqqoslanadi: faqat alohida chaqiruvlarni sanaydigan model va ketma-ket keladigan chaqiruvlar juftligini ham hisobga oladigan model. Dastur amaliy mashg'ulotning A8 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import random
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

tasodif = random.Random(1)

# Har ikki guruhda uchraydigan umumiy chaqiruvlar
umumiy = ["CreateFile", "ReadFile", "WriteFile", "CloseHandle", "RegOpenKey",
          "RegQueryValue", "LoadLibrary", "GetProcAddress", "Sleep"]
# Boshqa jarayonga kod kiritish zanjiri (zararli dasturlarga xos tartib)
kiritish = ["OpenProcess", "VirtualAllocEx", "WriteProcessMemory", "CreateRemoteThread"]

def iz(zararli):
    """Bitta jarayonning API chaqiruvlari ketma-ketligi (sun'iy ma'lumot)."""
    qadamlar = [tasodif.choice(umumiy) for _ in range(tasodif.randint(30, 60))]
    # Zararsiz dasturlarning bir qismi (masalan, tuzatuvchi dasturlar) ham
    # shu funksiyalarni ishlatadi, lekin boshqa tartibda
    if zararli or tasodif.random() < 0.3:
        bloklar = kiritish[:] if zararli else tasodif.sample(kiritish, 4)
        joy = tasodif.randint(0, len(qadamlar))
        qadamlar[joy:joy] = bloklar
    if zararli and tasodif.random() < 0.5:          # buzg'unchi ba'zan avtoyuklashga yozadi
        qadamlar.append("RegSetValue")
    elif not zararli and tasodif.random() < 0.3:    # o'rnatuvchi dasturlar ham shunday qiladi
        qadamlar.append("RegSetValue")
    return " ".join(qadamlar)

izlar = [iz(False) for _ in range(1500)] + [iz(True) for _ in range(500)]
y = [0] * 1500 + [1] * 500
iz_o, iz_s, y_o, y_s = train_test_split(izlar, y, test_size=0.3, stratify=y, random_state=0)

for n in [(1, 1), (1, 2)]:
    vekt = CountVectorizer(ngram_range=n, token_pattern=r"\S+", lowercase=False)
    model = LogisticRegression(max_iter=2000).fit(vekt.fit_transform(iz_o), y_o)
    taxmin = model.predict(vekt.transform(iz_s))
    print(f"n-gram oralig'i {n}: aniqlik {precision_score(y_s, taxmin):.2f}, "
          f"to'liqlik {recall_score(y_s, taxmin):.2f}, F1 {f1_score(y_s, taxmin):.2f}")

nomlar = vekt.get_feature_names_out()
eng = model.coef_[0].argsort()[::-1][:5]
print("\nZararli deb hisoblashga eng ko'p ta'sir qilgan belgilar:")
for i in eng:
    print(f"  {nomlar[i]:<40} {model.coef_[0][i]:.2f}")

n-gram oralig'i (1, 1): aniqlik 0.57, to'liqlik 0.66, F1 0.61


n-gram oralig'i (1, 2): aniqlik 0.94, to'liqlik 1.00, F1 0.97

Zararli deb hisoblashga eng ko'p ta'sir qilgan belgilar:
  VirtualAllocEx WriteProcessMemory        3.14
  WriteProcessMemory CreateRemoteThread    3.02
  OpenProcess VirtualAllocEx               2.56
  CreateRemoteThread LoadLibrary           0.83
  GetProcAddress OpenProcess               0.78


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
n-gram oralig'i (1, 1): aniqlik 0.57, to'liqlik 0.66, F1 0.61
n-gram oralig'i (1, 2): aniqlik 0.94, to'liqlik 1.00, F1 0.97

Zararli deb hisoblashga eng ko'p ta'sir qilgan belgilar:
  VirtualAllocEx WriteProcessMemory        3.14
  WriteProcessMemory CreateRemoteThread    3.02
  OpenProcess VirtualAllocEx               2.56
  CreateRemoteThread LoadLibrary           0.83
  GetProcAddress OpenProcess               0.78
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 8-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.